# BerkJonesSelector

Scores each feature with the two-sided Mann-Whitney U test and sorts the
p-values. Under the null, the i-th smallest of N p-values follows
Beta(i, N − i + 1). Berk-Jones scores rank i by that distribution's CDF at
the observed p-value: the chance of an i-th smallest p-value this small.
It keeps every feature up to the rank where that chance is smallest. It
picks its own count.

- **Cost:** one rank test per feature, the same as Higher Criticism.
- **Strength:** exact null distribution of each sorted p-value; no normal
  approximation.
- **Weakness:** it searches only the smallest `alpha0` share of p-values,
  and assumes independent features.
- **Modes:** two classes only.

The CDF is computed in log form, so it stays finite when the p-values and
the CDF are far below 1e-308.

In [1]:
import numpy as np

from featsel.selectors import BerkJonesSelector, HigherCriticismSelector


def rare_weak(n_signal, shift, n_samples=300, n_features=2000, seed=0):
    """Two classes; the first n_signal features differ in mean by shift."""
    rng = np.random.default_rng(seed)
    y = np.repeat([0, 1], n_samples // 2)
    X = rng.normal(size=(n_samples, n_features))
    X[y == 1, :n_signal] += shift
    return X, y


def report(selector, X, y, n_signal):
    chosen = set(selector.fit(X, y).get_support(indices=True))
    truth = set(range(n_signal))
    print(f'{selector!r:42s} kept {len(chosen):4d}: '
          f'{len(chosen & truth)} of {n_signal} signal, {len(chosen - truth)} false')

## Rare, weak signal

40 of 2,000 features carry a small shift. The three thresholds differ in
how far down the sorted p-values they go. Below, Berk-Jones keeps 25
features, 21 of them signal: fewer than HC2008, which keeps most of the
signal and many false features, and more than HC2004, which stops at the
single strongest feature.

In [2]:
X, y = rare_weak(n_signal=40, shift=0.4)
for selector in (BerkJonesSelector(), HigherCriticismSelector(variant='2008'),
                 HigherCriticismSelector(variant='2004')):
    report(selector, X, y, 40)

BerkJonesSelector()                        kept   25: 21 of 40 signal, 4 false
HigherCriticismSelector()                  kept   79: 34 of 40 signal, 45 false
HigherCriticismSelector(variant='2004')    kept    1: 1 of 40 signal, 0 false


## The score by rank

Berk-Jones compares each sorted p-value with its own null distribution.
Below, the score of the first ten ranks, as a log10 of the CDF. It falls
as more small p-values pile up than chance allows, and keeps falling past
rank 10 here. The rank where it is smallest sets the count. When one
p-value is far below all the others, rank 1 can win instead, and
Berk-Jones then keeps a single feature.

In [3]:
from featsel.selectors.berk_jones import log_beta_cdf

selector = BerkJonesSelector().fit(X, y)
log_p = np.sort(selector.log_pvalues_)[:10]
i = np.arange(1, 11)
n = X.shape[1]
for rank, lp, score in zip(i, log_p, log_beta_cdf(log_p, i, n - i + 1)):
    print(f'rank {rank:2d}: log10 p = {lp / np.log(10):7.2f}, '
          f'log10 BJ = {score / np.log(10):7.2f}')

rank  1: log10 p =   -6.54, log10 BJ =   -3.24
rank  2: log10 p =   -4.84, log10 BJ =   -3.38
rank  3: log10 p =   -4.53, log10 BJ =   -4.48
rank  4: log10 p =   -4.20, log10 BJ =   -5.01
rank  5: log10 p =   -4.08, log10 BJ =   -6.02
rank  6: log10 p =   -3.97, log10 BJ =   -6.93
rank  7: log10 p =   -3.89, log10 BJ =   -7.95
rank  8: log10 p =   -3.87, log10 BJ =   -9.28
rank  9: log10 p =   -3.86, log10 BJ =  -10.66
rank 10: log10 p =   -3.67, log10 BJ =  -10.43


## Only the order matters

A monotone transform of every feature changes nothing.

In [4]:
raw = BerkJonesSelector().fit(X, y).get_support()
transformed = BerkJonesSelector().fit(np.exp(X) ** 3, y).get_support()
print('same selection:', np.array_equal(raw, transformed))

same selection: True


## Widespread signal: the alpha0 edge

When signal is not rare, the score can keep falling up to the edge of the
search range, the smallest `alpha0` = 10% of p-values. Then the edge, not
the data, sets the count. Here 600 of 2,000 features carry a strong shift.

In [5]:
X_dense, y_dense = rare_weak(n_signal=600, shift=1.0)
report(BerkJonesSelector(), X_dense, y_dense, 600)
report(HigherCriticismSelector(), X_dense, y_dense, 600)

BerkJonesSelector()                        kept  200: 200 of 600 signal, 0 false
HigherCriticismSelector()                  kept  200: 200 of 600 signal, 0 false
